# Fusion Model Conversion

In [4]:
import os
import glob
import time
import tempfile
import warnings
import torch
import torchvision.transforms as transforms
import onnxruntime as ort
import numpy as np
import onnx
from onnxconverter_common import float16
from PIL import Image
from dotenv import load_dotenv
from transformers import ConvNextImageProcessor
from huggingface_hub import HfApi, hf_hub_download

# Import model loader from base_fusion_model.py
from base_fusion_model import load_fusion_model_from_hf

# Filter out PyTorch tracing and transformers load warnings for clean logs
warnings.filterwarnings("ignore", category=torch.jit.TracerWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ==========================================
# 1. PREPROCESSING PIPELINE SETUP
# ==========================================
IMG_SIZE = 260

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
])

eff_normalize = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def preprocess_image(pil_image: Image.Image, processor: ConvNextImageProcessor):
    resized_image = val_transforms(pil_image)
    pixel_eff = eff_normalize(resized_image).unsqueeze(0)
    enc = processor(images=resized_image, return_tensors='pt')
    pixel_cnx = enc['pixel_values']

    return pixel_eff, pixel_cnx


# ==========================================
# 2. CONVERT & UPLOAD TO HUGGING FACE
# ==========================================
def convert_and_upload_onnx(
    model: torch.nn.Module,
    processor: ConvNextImageProcessor,
    repo_id: str,
    onnx_filename_in_repo: str = "fusion_model.onnx",
    hf_token: str = None,
):
    model.eval()

    dummy_pil = Image.new("RGB", (IMG_SIZE, IMG_SIZE), color="white")
    dummy_eff, dummy_cnx = preprocess_image(dummy_pil, processor)

    with tempfile.TemporaryDirectory() as temp_dir:
        temp_onnx_path = os.path.join(temp_dir, onnx_filename_in_repo)

        print("Exporting Fusion Model to temporary ONNX graph...")
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            torch.onnx.export(
                model,
                (dummy_eff, dummy_cnx),
                temp_onnx_path,
                export_params=True,
                opset_version=18,
                do_constant_folding=True,
                input_names=["pixel_values_eff", "pixel_values_cnx"],
                output_names=["logits"],
                dynamic_axes={
                    "pixel_values_eff": {0: "batch_size"},
                    "pixel_values_cnx": {0: "batch_size"},
                    "logits": {0: "batch_size"},
                },
                dynamo=False,
            )
        print("✅ Base ONNX export successful.")

        # Convert ONNX graph weights to FP16 to keep file size ~142 MB
        print("Compressing ONNX model to FP16 precision...")
        onnx_model = onnx.load(temp_onnx_path)
        onnx_model_fp16 = float16.convert_float_to_float16(onnx_model, keep_io_types=True)
        onnx.save(onnx_model_fp16, temp_onnx_path)
        print("✅ ONNX graph compressed to FP16 successfully.")

        # Upload directly to Hugging Face Hub using token from .env
        api = HfApi(token=hf_token)
        print(f"Uploading '{onnx_filename_in_repo}' to Hugging Face repo: '{repo_id}'...")
        
        api.upload_file(
            path_or_fileobj=temp_onnx_path,
            path_in_repo=onnx_filename_in_repo,
            repo_id=repo_id,
            repo_type="model",
        )
        print(f"🚀 Successfully uploaded FP16 model to https://huggingface.co/{repo_id}/blob/main/{onnx_filename_in_repo}")


# ==========================================
# 3. BENCHMARK ON REAL TEST IMAGES
# ==========================================
def benchmark_fusion_with_folder(
    pytorch_model: torch.nn.Module,
    processor: ConvNextImageProcessor,
    repo_id: str,
    onnx_filename_in_repo: str,
    image_folder: str = r"D:\DamageLensAI\test_images",
    hf_token: str = None,
):
    print(f"Downloading '{onnx_filename_in_repo}' from Hugging Face for evaluation...")
    onnx_path = hf_hub_download(
        repo_id=repo_id, 
        filename=onnx_filename_in_repo,
        token=hf_token
    )
    session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

    supported_exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")
    image_paths = []
    for ext in supported_exts:
        image_paths.extend(glob.glob(os.path.join(image_folder, ext)))

    if not image_paths:
        raise FileNotFoundError(
            f"❌ No valid images found in folder '{image_folder}'. "
            f"Please place your test images inside '{image_folder}'."
        )

    print(f"📸 Found {len(image_paths)} images in '{image_folder}'. Starting benchmark...")

    pytorch_latencies = []
    onnx_latencies = []
    matches = 0

    # Warmup
    first_img = Image.open(sorted(image_paths)[0]).convert("RGB")
    warmup_eff, warmup_cnx = preprocess_image(first_img, processor)
    warmup_eff_np = warmup_eff.numpy()
    warmup_cnx_np = warmup_cnx.numpy()

    pytorch_model.eval()
    with torch.no_grad():
        _ = pytorch_model(warmup_eff, warmup_cnx)
    session.run(None, {"pixel_values_eff": warmup_eff_np, "pixel_values_cnx": warmup_cnx_np})

    # Benchmark loop
    print("\n" + "=" * 75)
    print(f"{'IMAGE NAME':<25} | {'PYTORCH (ms)':<14} | {'ONNX (ms)':<14} | {'MATCH?':<10}")
    print("=" * 75)

    for img_path in sorted(image_paths):
        file_name = os.path.basename(img_path)
        raw_img = Image.open(img_path).convert("RGB")
        
        pixel_values_eff, pixel_values_cnx = preprocess_image(raw_img, processor)
        np_eff = pixel_values_eff.numpy()
        np_cnx = pixel_values_cnx.numpy()

        # PyTorch Inference
        start_pt = time.perf_counter()
        with torch.no_grad():
            pt_logits = pytorch_model(pixel_values_eff, pixel_values_cnx)
        pt_time = (time.perf_counter() - start_pt) * 1000
        pytorch_latencies.append(pt_time)
        pt_pred = int(torch.argmax(pt_logits, dim=1)[0])

        # ONNX Runtime Inference
        start_onnx = time.perf_counter()
        onnx_outputs = session.run(
            None, 
            {"pixel_values_eff": np_eff, "pixel_values_cnx": np_cnx}
        )
        onnx_time = (time.perf_counter() - start_onnx) * 1000
        onnx_latencies.append(onnx_time)
        onnx_pred = int(np.argmax(onnx_outputs[0], axis=1)[0])

        is_match = pt_pred == onnx_pred
        if is_match:
            matches += 1
        match_str = "✅ YES" if is_match else "❌ NO"

        print(f"{file_name:<25} | {pt_time:<14.2f} | {onnx_time:<14.2f} | {match_str:<10}")

    avg_pt = np.mean(pytorch_latencies)
    avg_onnx = np.mean(onnx_latencies)
    speedup = ((avg_pt - avg_onnx) / avg_pt) * 100
    accuracy_match_pct = (matches / len(image_paths)) * 100

    print("=" * 75)
    print("                      BENCHMARK SUMMARY")
    print("=" * 75)
    print(f"Total Images Evaluated : {len(image_paths)}")
    print(f"Prediction Match Rate  : {accuracy_match_pct:.2f}% ({matches}/{len(image_paths)})")
    print(f"PyTorch Avg Latency    : {avg_pt:.2f} ms / image")
    print(f"ONNX Runtime Avg Latency: {avg_onnx:.2f} ms / image")
    print(f"Performance Gain       : {speedup:.2f}% faster with ONNX Runtime")
    print("=" * 75 + "\n")


# ==========================================
# MAIN EXECUTION FLOW
# ==========================================
if __name__ == "__main__":
    # Load environment variables from .env file
    load_dotenv()
    HF_TOKEN = os.getenv("HF_TOKEN")

    if not HF_TOKEN:
        print("⚠️ Warning: HF_TOKEN not found in .env file. Pushing/pulling private repos may fail.")

    NUM_CLASSES = 6
    HF_REPO_ID = "junaid17/best_fusion_model_fp16"
    ONNX_FILENAME = "fusion_model.onnx"
    TEST_IMAGE_FOLDER = r"D:\DamageLensAI\test_images"
    CONVNEXT_MODEL_NAME = "facebook/convnext-small-224"

    # Step 1: Initialize ConvNeXt Processor
    processor = ConvNextImageProcessor.from_pretrained(CONVNEXT_MODEL_NAME)

    # Step 2: Load PyTorch model from Hugging Face
    py_model = load_fusion_model_from_hf(
        repo_id=HF_REPO_ID,
        filename="best_fusion_model_fp16.pt",
        num_classes=NUM_CLASSES,
        device="cpu",
    )

    # Step 3: Convert to FP16 ONNX and push directly to Hugging Face
    convert_and_upload_onnx(
        model=py_model,
        processor=processor,
        repo_id=HF_REPO_ID,
        onnx_filename_in_repo=ONNX_FILENAME,
        hf_token=HF_TOKEN,
    )

    # Step 4: Run benchmarking on test image folder
    benchmark_fusion_with_folder(
        pytorch_model=py_model,
        processor=processor,
        repo_id=HF_REPO_ID,
        onnx_filename_in_repo=ONNX_FILENAME,
        image_folder=TEST_IMAGE_FOLDER,
        hf_token=HF_TOKEN,
    )

Loading weights: 100%|██████████| 342/342 [00:00<00:00, 1952.82it/s]
[transformers] ConvNextModel LOAD REPORT from: facebook/convnext-small-224
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully from Hugging Face.
Exporting Fusion Model to temporary ONNX graph...
✅ Base ONNX export successful.
Compressing ONNX model to FP16 precision...
✅ ONNX graph compressed to FP16 successfully.
Uploading 'fusion_model.onnx' to Hugging Face repo: 'junaid17/best_fusion_model_fp16'...


Processing Files (1 / 1): 100%|██████████|  142MB /  142MB,  680kB/s  
New Data Upload: 100%|██████████|  142MB /  142MB,  680kB/s  


🚀 Successfully uploaded FP16 model to https://huggingface.co/junaid17/best_fusion_model_fp16/blob/main/fusion_model.onnx
📸 Found 60 images in 'D:\DamageLensAI\test_images'. Starting benchmark...

IMAGE NAME                | PYTORCH (ms)   | ONNX (ms)      | MATCH?    
FB_10.jpg                 | 775.56         | 225.80         | ✅ YES     
FB_11.jpg                 | 683.06         | 263.21         | ✅ YES     
FB_13.jpg                 | 359.21         | 156.10         | ✅ YES     
FB_15.jpg                 | 357.64         | 152.04         | ✅ YES     
FB_16.jpg                 | 340.61         | 157.13         | ✅ YES     
FB_19.jpg                 | 343.34         | 157.87         | ✅ YES     
FB_20.jpg                 | 562.84         | 188.25         | ✅ YES     
FB_4.jpg                  | 328.80         | 162.34         | ✅ YES     
FB_5.jpg                  | 466.30         | 166.86         | ✅ YES     
FB_9.jpg                  | 657.23         | 188.45         | ✅ YES     
F

# Resnet Model Conversion

In [12]:
import os
import glob
import time
import tempfile
import warnings
import torch
import torchvision.transforms as transforms
import onnxruntime as ort
import numpy as np
import onnx
from onnxconverter_common import float16
from PIL import Image
from dotenv import load_dotenv
from huggingface_hub import HfApi, hf_hub_download

# Import ResNet loader
from base_model_resnet import load_resnet_model_from_hf

# Suppress PyTorch tracing and deprecation warnings
warnings.filterwarnings("ignore", category=torch.jit.TracerWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ==========================================
# 1. PREPROCESSING PIPELINE
# ==========================================
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def preprocess_image(pil_image: Image.Image) -> torch.Tensor:
    """
    Applies standard ResNet preprocessing and adds a batch dimension.
    Shape returned: (1, 3, 224, 224)
    """
    return test_transforms(pil_image).unsqueeze(0)


# ==========================================
# 2. CONVERT TO FP16 ONNX & UPLOAD
# ==========================================
def convert_and_upload_onnx_fp16(
    model: torch.nn.Module,
    repo_id: str,
    onnx_filename_in_repo: str = "car-damage-classifier.onnx",
    hf_token: str = None,
):
    """
    Exports PyTorch ResNet model to ONNX, compresses it to FP16,
    and pushes it directly to Hugging Face Hub.
    """
    model.eval()

    # Create dummy tensor for graph tracing (1, 3, 224, 224)
    dummy_input = torch.randn(1, 3, 224, 224)

    with tempfile.TemporaryDirectory() as temp_dir:
        temp_onnx_path = os.path.join(temp_dir, onnx_filename_in_repo)

        print("Exporting ResNet-18 Model to temporary FP32 ONNX graph...")
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            torch.onnx.export(
                model,
                dummy_input,
                temp_onnx_path,
                export_params=True,
                opset_version=18,
                do_constant_folding=True,
                input_names=["pixel_values"],
                output_names=["logits"],
                dynamic_axes={
                    "pixel_values": {0: "batch_size"},
                    "logits": {0: "batch_size"},
                },
                dynamo=False,
            )
        print("✅ Base ONNX export successful.")

        # Convert ONNX graph weights to FP16 to reduce file size
        print("Compressing ONNX model to FP16 precision...")
        onnx_model = onnx.load(temp_onnx_path)
        onnx_model_fp16 = float16.convert_float_to_float16(onnx_model, keep_io_types=True)
        onnx.save(onnx_model_fp16, temp_onnx_path)
        print("✅ ONNX graph compressed to FP16 successfully.")

        # Upload directly to Hugging Face Hub
        api = HfApi(token=hf_token)
        print(f"Uploading '{onnx_filename_in_repo}' to Hugging Face repo: '{repo_id}'...")
        
        api.upload_file(
            path_or_fileobj=temp_onnx_path,
            path_in_repo=onnx_filename_in_repo,
            repo_id=repo_id,
            repo_type="model",
        )
        print(f"🚀 Successfully uploaded FP16 ONNX model to https://huggingface.co/{repo_id}/blob/main/{onnx_filename_in_repo}")


# ==========================================
# 3. BENCHMARK ON REAL TEST IMAGES
# ==========================================
def benchmark_resnet_with_folder(
    pytorch_model: torch.nn.Module,
    repo_id: str,
    onnx_filename_in_repo: str,
    image_folder: str = r"D:\DamageLensAI\test_images",
    hf_token: str = None,
):
    """
    Downloads ONNX model from Hugging Face Hub and evaluates PyTorch vs ONNX Runtime performance.
    """
    print(f"Downloading '{onnx_filename_in_repo}' from Hugging Face for evaluation...")
    onnx_path = hf_hub_download(
        repo_id=repo_id, 
        filename=onnx_filename_in_repo,
        token=hf_token
    )
    session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

    supported_exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")
    image_paths = []
    for ext in supported_exts:
        image_paths.extend(glob.glob(os.path.join(image_folder, ext)))

    if not image_paths:
        raise FileNotFoundError(
            f"❌ No valid images found in folder '{image_folder}'. "
            f"Please verify your path."
        )

    print(f"📸 Found {len(image_paths)} images in '{image_folder}'. Starting benchmark...")

    pytorch_latencies = []
    onnx_latencies = []
    matches = 0

    # Warmup run
    first_img = Image.open(sorted(image_paths)[0]).convert("RGB")
    warmup_tensor = preprocess_image(first_img)
    warmup_np = warmup_tensor.numpy()

    pytorch_model.eval()
    with torch.no_grad():
        _ = pytorch_model(warmup_tensor)
    session.run(None, {"pixel_values": warmup_np})

    # Benchmark Loop
    print("\n" + "=" * 75)
    print(f"{'IMAGE NAME':<25} | {'PYTORCH (ms)':<14} | {'ONNX (ms)':<14} | {'MATCH?':<10}")
    print("=" * 75)

    for img_path in sorted(image_paths):
        file_name = os.path.basename(img_path)
        raw_img = Image.open(img_path).convert("RGB")
        
        img_tensor = preprocess_image(raw_img)
        np_input = img_tensor.numpy()

        # --- PyTorch Inference ---
        start_pt = time.perf_counter()
        with torch.no_grad():
            pt_logits = pytorch_model(img_tensor)
        pt_time = (time.perf_counter() - start_pt) * 1000
        pytorch_latencies.append(pt_time)
        pt_pred = int(torch.argmax(pt_logits, dim=1)[0])

        # --- ONNX Runtime Inference ---
        start_onnx = time.perf_counter()
        onnx_outputs = session.run(None, {"pixel_values": np_input})
        onnx_time = (time.perf_counter() - start_onnx) * 1000
        onnx_latencies.append(onnx_time)
        onnx_pred = int(np.argmax(onnx_outputs[0], axis=1)[0])

        is_match = pt_pred == onnx_pred
        if is_match:
            matches += 1
        match_str = "✅ YES" if is_match else "❌ NO"

        print(f"{file_name:<25} | {pt_time:<14.2f} | {onnx_time:<14.2f} | {match_str:<10}")

    avg_pt = np.mean(pytorch_latencies)
    avg_onnx = np.mean(onnx_latencies)
    speedup = ((avg_pt - avg_onnx) / avg_pt) * 100
    accuracy_match_pct = (matches / len(image_paths)) * 100

    print("=" * 75)
    print("                      BENCHMARK SUMMARY")
    print("=" * 75)
    print(f"Total Images Evaluated : {len(image_paths)}")
    print(f"Prediction Match Rate  : {accuracy_match_pct:.2f}% ({matches}/{len(image_paths)})")
    print(f"PyTorch Avg Latency    : {avg_pt:.2f} ms / image")
    print(f"ONNX Runtime Avg Latency: {avg_onnx:.2f} ms / image")
    print(f"Performance Gain       : {speedup:.2f}% faster with ONNX Runtime")
    print("=" * 75 + "\n")


# ==========================================
# MAIN EXECUTION FLOW
# ==========================================
if __name__ == "__main__":
    # Load token from .env
    load_dotenv()
    HF_TOKEN = os.getenv("HF_TOKEN")

    if not HF_TOKEN:
        print("⚠️ Warning: HF_TOKEN not found in .env file. Pushing/pulling private repos may fail.")

    NUM_CLASSES = 6
    HF_REPO_ID = "junaid17/car-damage-classifier"
    MODEL_FILENAME = "car-damage-classifier.pt"
    ONNX_FILENAME = "car-damage-classifier.onnx"
    TEST_IMAGE_FOLDER = r"D:\DamageLensAI\test_images"

    # Step 1: Load PyTorch ResNet Model from Hugging Face
    py_model = load_resnet_model_from_hf(
        repo_id=HF_REPO_ID,
        filename=MODEL_FILENAME,
        num_classes=NUM_CLASSES,
        device="cpu",
        hf_token=HF_TOKEN,
    )

    # Step 2: Convert to FP16 ONNX & Push directly to Hugging Face
    convert_and_upload_onnx_fp16(
        model=py_model,
        repo_id=HF_REPO_ID,
        onnx_filename_in_repo=ONNX_FILENAME,
        hf_token=HF_TOKEN,
    )

    # Step 3: Run benchmarking on test image folder
    benchmark_resnet_with_folder(
        pytorch_model=py_model,
        repo_id=HF_REPO_ID,
        onnx_filename_in_repo=ONNX_FILENAME,
        image_folder=TEST_IMAGE_FOLDER,
        hf_token=HF_TOKEN,
    )

✅ ResNet-18 model loaded successfully.
Exporting ResNet-18 Model to temporary FP32 ONNX graph...
✅ Base ONNX export successful.
Compressing ONNX model to FP16 precision...
✅ ONNX graph compressed to FP16 successfully.
Uploading 'car-damage-classifier.onnx' to Hugging Face repo: 'junaid17/car-damage-classifier'...


Processing Files (1 / 1): 100%|██████████| 22.6MB / 22.6MB, 1.05MB/s  
New Data Upload: 100%|██████████| 22.6MB / 22.6MB, 1.05MB/s  


🚀 Successfully uploaded FP16 ONNX model to https://huggingface.co/junaid17/car-damage-classifier/blob/main/car-damage-classifier.onnx
📸 Found 60 images in 'D:\DamageLensAI\test_images'. Starting benchmark...

IMAGE NAME                | PYTORCH (ms)   | ONNX (ms)      | MATCH?    
FB_10.jpg                 | 46.78          | 14.66          | ✅ YES     
FB_11.jpg                 | 56.13          | 15.03          | ✅ YES     
FB_13.jpg                 | 45.83          | 15.16          | ✅ YES     
FB_15.jpg                 | 44.06          | 14.06          | ✅ YES     
FB_16.jpg                 | 54.86          | 15.04          | ✅ YES     
FB_19.jpg                 | 61.18          | 15.78          | ✅ YES     
FB_20.jpg                 | 47.12          | 13.44          | ✅ YES     
FB_4.jpg                  | 49.21          | 17.78          | ✅ YES     
FB_5.jpg                  | 74.55          | 15.02          | ✅ YES     
FB_9.jpg                  | 48.68          | 14.77          |

# YOLO Model Conversion

In [11]:
import os
import glob
import time
import warnings
import torch
import torchvision.transforms as transforms
import onnxruntime as ort
import numpy as np
from PIL import Image
from dotenv import load_dotenv
from huggingface_hub import HfApi, hf_hub_download
from ultralytics import YOLO

# Import YOLO loader
from base_model_yolo import load_yolo_model_from_hf

# Suppress warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. PREPROCESSING PIPELINE
# ==========================================
IMG_SIZE = 640

yolo_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

def preprocess_image(pil_image: Image.Image) -> torch.Tensor:
    """Preprocesses PIL image for YOLO11 (1, 3, 640, 640)."""
    return yolo_transforms(pil_image).unsqueeze(0)


# ==========================================
# 2. CONVERT, OPTIMIZE TO FP16 & UPLOAD
# ==========================================
def convert_and_upload_yolo_onnx(
    yolo_model: YOLO,
    repo_id: str,
    onnx_filename_in_repo: str = "damage_detector.onnx",
    hf_token: str = None,
    imgsz: int = 640,
):
    print(f"Exporting YOLO11 Model directly to FP16 ONNX format (imgsz={imgsz})...")
    
    # Native Ultralytics FP16 export - lightning fast and avoids converter bugs
    exported_onnx_path = yolo_model.export(
        format="onnx",
        imgsz=imgsz,
        half=True,       # Shrinks the model to ~50MB natively
        dynamic=False,   # Disabling dynamic fixes the CPU type-mismatch errors
        simplify=True,
        opset=17
    )
    print("✅ FP16 ONNX export successful.")

    file_size_mb = os.path.getsize(exported_onnx_path) / (1024 * 1024)
    print(f"📦 Model File Size: {file_size_mb:.2f} MB")

    # Upload to Hugging Face Hub
    api = HfApi(token=hf_token)
    print(f"Uploading '{onnx_filename_in_repo}' to Hugging Face repo: '{repo_id}'...")
    
    api.upload_file(
        path_or_fileobj=exported_onnx_path,
        path_in_repo=onnx_filename_in_repo,
        repo_id=repo_id,
        repo_type="model",
    )
    print(f"🚀 Successfully uploaded ONNX model to https://huggingface.co/{repo_id}/blob/main/{onnx_filename_in_repo}")


# ==========================================
# 3. BENCHMARK ON REAL TEST IMAGES
# ==========================================
def benchmark_yolo_with_folder(
    yolo_model: YOLO,
    repo_id: str,
    onnx_filename_in_repo: str,
    image_folder: str = r"D:\DamageLensAI\test_images",
    hf_token: str = None,
    conf_thresh: float = 0.25,
):
    print(f"Downloading '{onnx_filename_in_repo}' from Hugging Face for evaluation...")
    
    # force_download=True ensures we grab the newly uploaded 50MB file
    onnx_path = hf_hub_download(
        repo_id=repo_id, 
        filename=onnx_filename_in_repo,
        token=hf_token,
        force_download=True 
    )
    
    session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name

    supported_exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")
    image_paths = []
    for ext in supported_exts:
        image_paths.extend(glob.glob(os.path.join(image_folder, ext)))

    if not image_paths:
        raise FileNotFoundError(f"❌ No valid images found in folder '{image_folder}'.")

    print(f"📸 Found {len(image_paths)} images in '{image_folder}'. Starting benchmark...")

    pytorch_latencies = []
    onnx_latencies = []
    matches = 0

    # Warmup
    first_img = Image.open(sorted(image_paths)[0]).convert("RGB")
    warmup_tensor = preprocess_image(first_img)
    warmup_np = warmup_tensor.numpy()

    _ = yolo_model.model(warmup_tensor)
    session.run(None, {input_name: warmup_np})

    print("\n" + "=" * 75)
    print(f"{'IMAGE NAME':<25} | {'PYTORCH (ms)':<14} | {'ONNX (ms)':<14} | {'MATCH?':<10}")
    print("=" * 75)

    for img_path in sorted(image_paths):
        file_name = os.path.basename(img_path)
        raw_img = Image.open(img_path).convert("RGB")
        
        img_tensor = preprocess_image(raw_img)
        np_input = img_tensor.numpy()

        start_pt = time.perf_counter()
        with torch.no_grad():
            pt_out = yolo_model.model(img_tensor)
        pt_time = (time.perf_counter() - start_pt) * 1000
        pytorch_latencies.append(pt_time)

        start_onnx = time.perf_counter()
        onnx_out = session.run(None, {input_name: np_input})
        onnx_time = (time.perf_counter() - start_onnx) * 1000
        onnx_latencies.append(onnx_time)

        pt_preds = pt_out[0] if isinstance(pt_out, (list, tuple)) else pt_out
        pt_max_conf = float(torch.max(pt_preds[0, 4, :])) if pt_preds.ndim == 3 else 0.0
        
        onnx_preds = onnx_out[0]
        onnx_max_conf = float(np.max(onnx_preds[0, 4, :])) if onnx_preds.ndim == 3 else 0.0

        pt_detected = pt_max_conf >= conf_thresh
        onnx_detected = onnx_max_conf >= conf_thresh
        is_match = (pt_detected == onnx_detected)

        if is_match:
            matches += 1
        match_str = "✅ YES" if is_match else "❌ NO"

        print(f"{file_name:<25} | {pt_time:<14.2f} | {onnx_time:<14.2f} | {match_str:<10}")

    avg_pt = np.mean(pytorch_latencies)
    avg_onnx = np.mean(onnx_latencies)
    speedup = ((avg_pt - avg_onnx) / avg_pt) * 100
    accuracy_match_pct = (matches / len(image_paths)) * 100

    print("=" * 75)
    print("                      BENCHMARK SUMMARY")
    print("=" * 75)
    print(f"Total Images Evaluated : {len(image_paths)}")
    print(f"Detection Match Rate   : {accuracy_match_pct:.2f}% ({matches}/{len(image_paths)})")
    print(f"PyTorch Avg Latency    : {avg_pt:.2f} ms / image")
    print(f"ONNX Runtime Avg Latency: {avg_onnx:.2f} ms / image")
    print(f"Performance Gain       : {speedup:.2f}% faster with ONNX Runtime")
    print("=" * 75 + "\n")


# ==========================================
# MAIN EXECUTION FLOW
# ==========================================
if __name__ == "__main__":
    load_dotenv()
    HF_TOKEN = os.getenv("HF_TOKEN")

    if not HF_TOKEN:
        print("⚠️ Warning: HF_TOKEN not found in .env file.")

    HF_REPO_ID = "junaid17/Yolo_Model"
    MODEL_FILENAME = "damage_detector.pt"
    ONNX_FILENAME = "damage_detector.onnx"
    TEST_IMAGE_FOLDER = r"D:\DamageLensAI\test_images"

    yolo_model = load_yolo_model_from_hf(
        repo_id=HF_REPO_ID,
        filename=MODEL_FILENAME,
        hf_token=HF_TOKEN,
    )

    convert_and_upload_yolo_onnx(
        yolo_model=yolo_model,
        repo_id=HF_REPO_ID,
        onnx_filename_in_repo=ONNX_FILENAME,
        hf_token=HF_TOKEN,
        imgsz=IMG_SIZE,
    )

    benchmark_yolo_with_folder(
        yolo_model=yolo_model,
        repo_id=HF_REPO_ID,
        onnx_filename_in_repo=ONNX_FILENAME,
        image_folder=TEST_IMAGE_FOLDER,
        hf_token=HF_TOKEN,
    )

✅ YOLO model loaded successfully.
Exporting YOLO11 Model directly to FP16 ONNX format (imgsz=640)...
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.116  Python-3.11.0 torch-2.13.0+cpu CPU (12th Gen Intel Core i3-1215U)
YOLO11l summary (fused): 190 layers, 25,280,083 parameters, 0 gradients, 86.8 GFLOPs

PyTorch: starting from 'C:\Users\junai\.cache\huggingface\hub\models--junaid17--Yolo_Model\snapshots\0cbc6f112ebc6cb7df05fe93f3f0175cc847bec0\damage_detector.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (48.8 MB)

ONNX: starting export with onnx 1.22.0 opset 17...
ONNX: slimming with onnxslim 0.1.95...
ONNX: converting to FP16...
ONNX: export success  7.1s, saved as 'C:\Users\junai\.cache\huggingface\hub\models--junaid17--Yolo_Model\snapshots\0cbc6f112ebc6cb7df05fe93f3f0175cc847bec0\damage_detector.onnx' (48.5 MB)

Export complete (11.6s)
Results saved to C:\Users\junai\.cache\huggingface\hub\models-

Processing Files (1 / 1): 100%|██████████| 50.9MB / 50.9MB, 2.79MB/s  
New Data Upload: 100%|██████████| 8.84MB / 8.84MB,  568kB/s  


🚀 Successfully uploaded ONNX model to https://huggingface.co/junaid17/Yolo_Model/blob/main/damage_detector.onnx
📸 Found 60 images in 'D:\DamageLensAI\test_images'. Starting benchmark...

IMAGE NAME                | PYTORCH (ms)   | ONNX (ms)      | MATCH?    
FB_10.jpg                 | 1961.16        | 933.21         | ✅ YES     
FB_11.jpg                 | 1645.58        | 821.06         | ✅ YES     
FB_13.jpg                 | 1973.07        | 884.90         | ✅ YES     
FB_15.jpg                 | 2296.55        | 1064.76        | ✅ YES     
FB_16.jpg                 | 1159.60        | 679.19         | ✅ YES     
FB_19.jpg                 | 1310.77        | 729.79         | ✅ YES     
FB_20.jpg                 | 1263.96        | 678.22         | ✅ YES     
FB_4.jpg                  | 1385.82        | 658.67         | ✅ YES     
FB_5.jpg                  | 1352.15        | 665.79         | ✅ YES     
FB_9.jpg                  | 1281.14        | 654.91         | ✅ YES     
FC_1.jpg  